In [1]:
!pip install geopandas

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

drive_path = '/content/drive/MyDrive/2nd SeD/F4/data/(B100)국토통계_인구정보-생산가능 인구 수(전체)-(격자) 100M_경상북도 영주시_202410'
# Check if the directory exists
if os.path.exists(drive_path):
    print(f"Listing files in: {drive_path}")
    for root, dirs, files in os.walk(drive_path):
        for file in files:
            print(os.path.join(root, file))
else:
    print(f"Directory not found: {drive_path}. Please check the path and try again.")


Listing files in: /content/drive/MyDrive/2nd SeD/F4/data/(B100)국토통계_인구정보-생산가능 인구 수(전체)-(격자) 100M_경상북도 영주시_202410
/content/drive/MyDrive/2nd SeD/F4/data/(B100)국토통계_인구정보-생산가능 인구 수(전체)-(격자) 100M_경상북도 영주시_202410/vl_blk.prj
/content/drive/MyDrive/2nd SeD/F4/data/(B100)국토통계_인구정보-생산가능 인구 수(전체)-(격자) 100M_경상북도 영주시_202410/vl_blk.shx
/content/drive/MyDrive/2nd SeD/F4/data/(B100)국토통계_인구정보-생산가능 인구 수(전체)-(격자) 100M_경상북도 영주시_202410/vl_blk.dbf
/content/drive/MyDrive/2nd SeD/F4/data/(B100)국토통계_인구정보-생산가능 인구 수(전체)-(격자) 100M_경상북도 영주시_202410/vl_blk.shp
/content/drive/MyDrive/2nd SeD/F4/data/(B100)국토통계_인구정보-생산가능 인구 수(전체)-(격자) 100M_경상북도 영주시_202410/vl_blk.cst


In [4]:
import geopandas as gpd

# The .shp file is the main geometry file for a Shapefile
shp_file_path = '/content/drive/MyDrive/2nd SeD/F4/data/(B100)국토통계_인구정보-생산가능 인구 수(전체)-(격자) 100M_경상북도 영주시_202410/vl_blk.shp'

# Load the Shapefile using geopandas
try:
    gdf = gpd.read_file(shp_file_path)
    print("Shapefile이 성공적으로 로드되었습니다.")
    # Display the first 5 rows of the GeoDataFrame
    display(gdf.head())
except Exception as e:
    print(f"Shapefile 로드 중 오류가 발생했습니다: {e}")

Shapefile이 성공적으로 로드되었습니다.


,gid,lbl,val,geometry
0,ë¼ë°823757,None,NaN,"POLYGON ((1082300 1875700, 1082300 1875800, 10..."
1,ë¼ë°823758,None,NaN,"POLYGON ((1082300 1875800, 1082300 1875900, 10..."
2,ë¼ë°823759,None,NaN,"POLYGON ((1082300 1875900, 1082300 1876000, 10..."
3,ë¼ë°823760,None,NaN,"POLYGON ((1082300 1876000, 1082300 1876100, 10..."
4,ë¼ë°824757,None,NaN,"POLYGON ((1082400 1875700, 1082400 1875800, 10..."


https://map.ngii.go.kr/ms/map/NlipMap.do
출처 : 국토정보플랫폼_국토정보맵

경상북도 영주시 100m 격자별 생산가능 인구 수

위 지도는 영주시의 격자별 생산가능 인구 현황을 시각화한 것입니다. 각 격자의 색상 진하기는 인구 밀도를 나타내며, `YlOrRd` 색상 맵은 노란색에서 붉은색으로 갈수록 인구 밀도가 높음을 의미합니다. 축 레이블은 지도 가독성을 위해 비활성화했습니다.

In [5]:
import folium
import pandas as pd
from branca.colormap import linear

# Reproject to WGS84 (latitude/longitude) for Folium
# Assuming the original CRS might be EPSG:5179 (Korea 2000 / Central Belt) if not explicitly set.
# geopandas' read_file usually infers CRS from .prj file, so gdf.crs should be set.
# If gdf.crs is None, we'll try to set it to EPSG:5179 before reprojecting.
if gdf.crs is None:
    try:
        gdf_wgs84 = gdf.set_crs(epsg=5179, allow_override=True).to_crs(epsg=4326)
        print("CRS가 명시되지 않아 EPSG:5179로 설정 후 WGS84로 재투영했습니다.")
    except Exception as e:
        print(f"경고: CRS 설정 또는 재투영 중 오류 발생: {e}. 원본 CRS 추론에 따라 진행합니다.")
        gdf_wgs84 = gdf.to_crs(epsg=4326) # Fallback to reproject directly
else:
    gdf_wgs84 = gdf.to_crs(epsg=4326)

# Calculate the centroid of the GeoDataFrame for centering the map
# Using union_all() as unary_union is deprecated
map_center = gdf_wgs84.union_all().centroid
map_latitude = map_center.y
map_longitude = map_center.x

# Create a Folium map
m = folium.Map(location=[map_latitude, map_longitude], zoom_start=12)

# Check if 'val' column has any non-NaN values to decide if choropleth is possible
if gdf_wgs84['val'].notna().any():
    min_val = gdf_wgs84['val'].min()
    max_val = gdf_wgs84['val'].max()

    # Create a colormap (using YlOrRd_09 as in the previous matplotlib plot)
    colormap = linear.YlOrRd_09.scale(vmin=min_val, vmax=max_val)
    colormap.caption = '생산가능 인구 수'
    m.add_child(colormap)

    def style_function(feature):
        value = feature['properties']['val']
        if pd.isna(value):
            return {
                'fillColor': '#cccccc', # Grey for NaN values
                'color': 'black',
                'weight': 0.5,
                'fillOpacity': 0.5
            }
        else:
            return {
                'fillColor': colormap(value),
                'color': 'black',
                'weight': 0.5,
                'fillOpacity': 0.7
            }

    folium.GeoJson(
        gdf_wgs84,
        style_function=style_function,
        tooltip=folium.GeoJsonTooltip(fields=['val'], aliases=['생산가능 인구 수'], localize=True)
    ).add_to(m)

else:
    print("경고: 'val' 컬럼에 유효한 값이 없어 코로플레스 스타일을 적용할 수 없습니다. 기본 스타일로 지오메트리를 표시합니다.")
    folium.GeoJson(
        gdf_wgs84,
        style_function=lambda feature: {
            'fillColor': '#ADD8E6', # Light blue default fill
            'color': 'black',
            'weight': 0.5,
            'fillOpacity': 0.7
        },
        tooltip=folium.GeoJsonTooltip(fields=['gid'], aliases=['격자 ID'], localize=True) # Show GID in tooltip if no val
    ).add_to(m)

# Display the map
display(m)

Output hidden; open in https://colab.research.google.com to view.

https://bike.yeongju.go.kr/stn/stationState.do?index=0301
출처 : 영주시 공공자전거 무인대여 시스템
현재 존재하는 7개 터미널의 위치를 얻고 직접 좌표 도출

In [9]:
import pandas as pd

terminal_data = {
    '명칭': ['영주 자전거 공원', '무섬 마을', '한정교', '서천주차장', '선비촌', '용혈폭포', '술바위교차로'],
    '위도': [36.8246790791725, 36.7331590343526, 36.8003509948257, 36.8217430074986, 36.9284735311732, 36.718587745264, 36.8180154370523], # '술바위교차로' 위도 수정
    '경도': [128.61585862526, 128.621034728972, 128.614905949604, 128.613500655412, 128.58343367327, 128.653782582992, 128.637134542705]  # '술바위교차로' 경도 수정
}
terminals_df = pd.DataFrame(terminal_data)
print("구루미 터미널 데이터프레임 생성 완료.")
display(terminals_df.head())

구루미 터미널 데이터프레임 생성 완료.


,명칭,위도,경도
0,영주 자전거 공원,36.824679,128.615859
1,무섬 마을,36.733159,128.621035
2,한정교,36.800351,128.614906
3,서천주차장,36.821743,128.613501
4,선비촌,36.928474,128.583434


In [10]:
import folium
import pandas as pd

# 기존 버스 정류장 데이터 파일 경로
bus_stop_file_path = '/content/drive/MyDrive/2nd SeD/F4/data/경상북도 영주시_버스정류장 현황_20250731.csv'

try:
    # 버스 정류장 데이터 로드 (cp949 인코딩 지정)
    bus_stops_df = pd.read_csv(bus_stop_file_path, encoding='cp949')
    print("버스 정류장 데이터 로드 완료.")

    # terminals_df는 이전 셀에서 생성된 데이터프레임을 사용합니다.

    # 지도의 중심점 계산 (두 데이터셋의 평균 위도/경도 사용)
    all_latitudes = pd.concat([bus_stops_df['위도'], terminals_df['위도']])
    all_longitudes = pd.concat([bus_stops_df['경도'], terminals_df['경도']])

    map_center_lat = all_latitudes.mean()
    map_center_lon = all_longitudes.mean()

    # 새로운 Folium 지도 생성
    combined_map = folium.Map(location=[map_center_lat, map_center_lon], zoom_start=12)

    # ------------------- 버스 정류장 마커 추가 -------------------
    for idx, row in bus_stops_df.iterrows():
        tooltip_text = f"정류장명: {row.get('정류장명', '정보 없음')}<br>위도: {row['위도']}<br>경도: {row['경도']}"
        folium.Marker(
            location=[row['위도'], row['경도']],
            popup=folium.Popup(tooltip_text, max_width=300),
            tooltip=row.get('정류장명', f"({row['위도']:.2f}, {row['경도']:.2f})"),
            icon=folium.Icon(color='red') # 버스 정류장은 빨간색 마커
        ).add_to(combined_map)
    print("버스 정류장 마커 추가 완료.")

    # ------------------- 터미널 마커 추가 -------------------
    for idx, row in terminals_df.iterrows():
        tooltip_text = f"터미널명: {row.get('명칭', '정보 없음')}<br>위도: {row['위도']}<br>경도: {row['경도']}"
        folium.Marker(
            location=[row['위도'], row['경도']],
            popup=folium.Popup(tooltip_text, max_width=300),
            tooltip=row.get('명칭', f"({row['위도']:.2f}, {row['경도']:.2f})"),
            icon=folium.Icon(color='blue') # 터미널은 파란색 마커
        ).add_to(combined_map)
    print("터미널 마커 추가 완료.")

    print("버스 정류장과 터미널 통합 시각화가 완료되었습니다.")
    # 지도 표시
    display(combined_map)

except FileNotFoundError as e:
    print(f"오류: 파일을 찾을 수 없습니다. 경로를 확인해주세요: {e}")
except KeyError as e:
    print(f"오류: 데이터프레임에서 필요한 컬럼을 찾을 수 없습니다. 컬럼 이름이 올바른지 확인해주세요. 누락된 컬럼: {e}")
except Exception as e:
    print(f"데이터 시각화 중 오류가 발생했습니다: {e}")

버스 정류장 데이터 로드 완료.
버스 정류장 마커 추가 완료.
터미널 마커 추가 완료.
버스 정류장과 터미널 통합 시각화가 완료되었습니다.
